# DuckDB explorer

Connects to the local DuckDB file(s) this pipeline actually writes today
(`pipeline.duckdb_staging` — one `.duckdb` file per ingest run, under
`Settings.duckdb_staging_dir`) and lists every table found. There's no
single persistent warehouse file yet (that's `TODO.md`'s "Storage & Cost
Optimization" Phase 1) — each staging file holds exactly one table,
always named `landed` (`duckdb_staging.py`'s `_TABLE` constant), so this
mostly confirms that and reports how many files/rows are sitting there.


In [1]:
import sys
import os
from pathlib import Path
from dotenv import dotenv_values


# This notebook can live at `<repo>/notebooks/` or (as moved) at
# `<repo>/services/data-pipeline/notebooks/` -- search a few likely
# spots, then walk upward, instead of assuming a fixed relative offset.
def _find_data_pipeline_dir() -> Path:
    marker = Path("app") / "core" / "config.py"
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd() / "services" / "data-pipeline",
        Path.cwd().parent / "services" / "data-pipeline",
    ]
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / marker).exists():
            return candidate
    cur = Path.cwd().resolve()
    for _ in range(8):
        candidate = (cur / "services" / "data-pipeline").resolve()
        if (candidate / marker).exists():
            return candidate
        if cur.parent == cur:
            break
        cur = cur.parent
    raise FileNotFoundError(
        f"couldn't locate services/data-pipeline (with app/core/config.py) "
        f"starting from cwd={Path.cwd()}"
    )


DATA_PIPELINE_DIR = _find_data_pipeline_dir()
sys.path.insert(0, str(DATA_PIPELINE_DIR))
os.chdir(DATA_PIPELINE_DIR)

env_path = DATA_PIPELINE_DIR / ".env"
assert env_path.exists(), f"no .env at {env_path}"
for key, value in dotenv_values(env_path).items():
    if value is not None:
        os.environ[key] = value

# `app` only resolves once `sys.path` is patched above -- guarding with
# try/except ImportError (same convention as model-training.ipynb's
# setup cell) keeps this off ruff's E402 (module-level import not at
# top of cell), which doesn't flag imports inside try/except.
try:
    from app.core.config import get_settings

    get_settings.cache_clear()
except ImportError:
    pass

print("resolved services/data-pipeline at:", DATA_PIPELINE_DIR)

resolved services/data-pipeline at: /Users/macbook/Project/research/EcoLens/services/data-pipeline


In [2]:
import duckdb

settings = get_settings()
staging_dir = Path(settings.duckdb_staging_dir)
if not staging_dir.is_absolute():
    staging_dir = (DATA_PIPELINE_DIR / staging_dir).resolve()

duckdb_files = sorted(staging_dir.glob("*.duckdb"))
print(f"duckdb_staging_dir: {staging_dir}")
print(f"found {len(duckdb_files)} .duckdb file(s)\n")

all_tables: set[str] = set()
total_rows = 0

for path in duckdb_files:
    con = duckdb.connect(str(path), read_only=True)
    try:
        tables = [row[0] for row in con.execute("SHOW TABLES").fetchall()]
        all_tables.update(tables)
        for table in tables:
            total_rows += con.execute(
                f"SELECT count(*) FROM {table}"  # nosec B608 -- table name from this file's own SHOW TABLES, not external input
            ).fetchone()[0]
    finally:
        con.close()

print("all table names found across every .duckdb file:")
print(sorted(all_tables))
print(f"\ntotal rows across all staged files: {total_rows}")

duckdb_staging_dir: /Users/macbook/Project/research/EcoLens/services/data-pipeline/data/staging
found 0 .duckdb file(s)

all table names found across every .duckdb file:
[]

total rows across all staged files: 0
